# E1_en chạy lại với corpus cùng số token (T13)

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

Báo cáo viết "nhánh EN được cắt xuống đúng cùng số token với VI", nhưng số thật:
VI = 38.250.964, EN = 42.147.057 — cả hai nhỏ hơn budget 50M nên lệnh cắt cũ chưa
bao giờ chạy (xem `docs/04_task_sua_bao_cao.md`, mục T1/T13). Ở đây chạy lại 6 run
`E1_en` với corpus cắt đúng 38.250.964 token, budget giữ 50M — số bước và epoch
(~1,31) trùng với nhánh VI.

Phải lấy 110.000 bài thay vì 90.000 như E1 gốc: bản `datasets` trên Colab lấy mẫu
ra ít token hơn hồi chạy Kaggle, 90k bài chỉ được ~35,3M — thấp hơn đích thì không
có gì để cắt. Lấy dư rồi cắt, phần dư không ảnh hưởng vì mô hình chỉ thấy
38.250.964 token đầu.

6 run = HHHH/AAAA × seed 0–2, tốn ~12 phút token hoá (một lần, có cache) cộng
~1 giờ GPU T4. Nhớ bật GPU trước: Runtime → Change runtime type → T4.

Notebook chỉ clone repo và gọi `hyena_study`, không chứa logic thí nghiệm.


## 1 · Nạp mã nguồn

In [21]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/content/Hyena-Attention-Study"

import os, shutil, subprocess, sys

os.chdir("/content")             # thoát khỏi WORK trước khi xoá: rmtree thư mục
                                 # đang đứng làm git chết 128 "unable to read cwd"

if os.path.isdir(WORK):
    shutil.rmtree(WORK)          # chạy lại từ đầu -> luôn lấy bản mới nhất
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

print("Đã clone vào", WORK)


Đã clone vào /content/Hyena-Attention-Study


## 2 · Cài thư viện và kiểm tra GPU

In [22]:
!pip install -q datasets tokenizers

import torch, datasets

print("torch", torch.__version__, "· datasets", datasets.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHƯA BẬT GPU — Runtime -> Change runtime type -> GPU (T4), rồi chạy lại")
    print("!! 6 lần chạy 50M token trên CPU là bất khả thi")
    print("!" * 70)


torch 2.11.0+cu128 · datasets 4.0.0
GPU: Tesla T4 · 14.6 GB


## 3 · Kiểm trước khi đốt GPU

Ba bước, bước nào fail thì dừng: cờ `--max_train_tokens` phải có trên bản GitHub;
test đường ống chạy CPU khoảng một phút; dựng cache và xem tập train cắt ra đúng
38.250.964 chưa (~12 phút). Bước cuối chặn đúng lỗi đã gặp một lần: corpus kéo về
nhỏ hơn đích thì không có gì để cắt, chạy tiếp chỉ phí GPU.


In [23]:
import subprocess, sys

help_text = subprocess.run(
    [sys.executable, "-m", "hyena_study.train", "--help"],
    capture_output=True, text=True).stdout
assert "--max_train_tokens" in help_text, (
    "Bản trên GitHub CHƯA có cờ --max_train_tokens. "
    "Push commit T13 lên GitHub rồi chạy lại từ ô clone."
)
print("OK: cờ --max_train_tokens có mặt.")

OK: cờ --max_train_tokens có mặt.


In [24]:
!python tests/test_pipeline.py


  [PASS] P1a Token hoá âm tiết khứ hồi  (46)
  [PASS] P1b Chuẩn hoá NFC hợp nhất mã
  [PASS] P1c Số âm tiết đúng như mong đợi  (7)
  [PASS] P2a Đầu vào/nhãn lệch đúng 1  (9)
  [PASS] P2b Ngân sách token được tôn trọng  (5000)
  [PASS] P3a Hyena học được  (0.023)
  [PASS] P3b Transformer học được  (0.024)
  [PASS] P3c Mô hình lai học được  (0.023)
  [PASS] P3d Hyena học được KHÔNG cần pos-emb  (0.022)
  [PASS] P4a Đường ống alpha-corpus (E4)
  [PASS] P4b Lịch learning rate
  [PASS] P4c Cờ --max_train_tokens tồn tại
  [PASS] P4d Cắt corpus tách rời ngân sách bước

==> Toàn bộ 13 test ĐẠT


In [25]:
from hyena_study.data import cached_token_stream

train, val, test, tok, stats = cached_token_stream(
    lang="en", tokenizer="syllable", vocab_size=16000, n_docs=110000,
    data_seed=0, max_tokens=38250964, cache_root="data_cache",
)
print(f"train={len(train):,} val={len(val):,} test={len(test):,} "
      f"| vocab={tok.vocab_size:,} | unk={stats.unk_rate:.4f}")
assert len(train) == 38250964, (
    f"train={len(train):,} != 38.250.964 — 110000 bài vẫn chưa đủ token để cắt. "
    "Tăng n_docs ở Ô NÀY và trong 6 ô chạy bên dưới (cùng một giá trị!) rồi chạy lại."
)
print("OK: cache đã dựng, tập train cắt đúng 38.250.964 token.")

[cache] khong co san, dang dung data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

[cache] da luu 43,319,051 token train
train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046
OK: cache đã dựng, tập train cắt đúng 38.250.964 token.


## 4 · Sáu lần chạy

Tham số như E1 gốc, khác `--n_docs 110000` và `--max_train_tokens 38250964`.
Ghi vào `results_t13/`, tên run giữ `E1_en_*` để lát chép đè vào `results/`.
Cache đã dựng ở bước 3 nên mỗi run chỉ ~6–9 phút, chạy tuần tự từng ô, ô nào
lỗi chạy lại riêng ô đó.

Mỗi run để ý dòng `token: train=38,250,964` và `6.103 bước`.


In [26]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 0 --run_name E1_en_HHHH_s0 --out_dir results_t13


[E1_en_HHHH_s0] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_HHHH_s0] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_HHHH_s0] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E1_en_HHHH_s0] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.9255 | |g| 2.04 | 0.8M token | 9s | 91.1k tok/s
  b   200/6103 | loss 6.2986 | |g| 1.50 | 1.6M token | 17s | 96.6k tok/s
  b   300/6103 | loss 5.9538 | |g| 1.51 | 2.5M token | 25s | 98.1k tok/s
  b   400/6103 | loss 5.6671 | |g| 1.36 | 3.3M token | 33s | 98.8k tok/s
  b   500/6103 | loss 5.4871 | |g| 1.40 | 4.1M token | 41s | 99.3k tok/s
  --> val loss 5.3019 | val PPL 200.73
  b   600/6103 | loss 5.3455 | |g| 2.04 | 4.9M token | 52s | 94.1k tok/s
  b   700/6103 | loss 5.0503 | |g| 1.23 | 5.7M token | 61s | 94.5k tok/s
  b   800/6103 | loss 5.082

In [27]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 1 --run_name E1_en_HHHH_s1 --out_dir results_t13


[E1_en_HHHH_s1] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_HHHH_s1] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_HHHH_s1] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E1_en_HHHH_s1] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8359 | |g| 2.15 | 0.8M token | 10s | 85.8k tok/s
  b   200/6103 | loss 6.3090 | |g| 1.01 | 1.6M token | 18s | 90.4k tok/s
  b   300/6103 | loss 5.7325 | |g| 1.83 | 2.5M token | 27s | 91.6k tok/s
  b   400/6103 | loss 5.5695 | |g| 1.94 | 3.3M token | 36s | 92.1k tok/s
  b   500/6103 | loss 5.3362 | |g| 1.56 | 4.1M token | 44s | 92.4k tok/s
  --> val loss 5.2780 | val PPL 195.98
  b   600/6103 | loss 5.2590 | |g| 1.49 | 4.9M token | 56s | 87.7k tok/s
  b   700/6103 | loss 5.0624 | |g| 1.77 | 5.7M token | 65s | 88.7k tok/s
  b   800/6103 | loss 4.75

In [28]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 2 --run_name E1_en_HHHH_s2 --out_dir results_t13


[E1_en_HHHH_s2] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_HHHH_s2] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_HHHH_s2] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E1_en_HHHH_s2] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8535 | |g| 2.28 | 0.8M token | 10s | 85.5k tok/s
  b   200/6103 | loss 6.1880 | |g| 1.23 | 1.6M token | 18s | 90.1k tok/s
  b   300/6103 | loss 5.8799 | |g| 1.43 | 2.5M token | 27s | 91.4k tok/s
  b   400/6103 | loss 5.4677 | |g| 1.49 | 3.3M token | 36s | 91.9k tok/s
  b   500/6103 | loss 5.2775 | |g| 1.30 | 4.1M token | 44s | 92.3k tok/s
  --> val loss 5.2677 | val PPL 193.98
  b   600/6103 | loss 5.1685 | |g| 1.28 | 4.9M token | 56s | 87.7k tok/s
  b   700/6103 | loss 5.0957 | |g| 1.17 | 5.7M token | 65s | 88.7k tok/s
  b   800/6103 | loss 5.13

In [29]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 0 --run_name E1_en_AAAA_s0 --out_dir results_t13


[E1_en_AAAA_s0] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_AAAA_s0] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_AAAA_s0] tham số: 7,383,552 (mixer 1,049,600 · mlp 2,102,272 · emb 4,096,000)
[E1_en_AAAA_s0] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.9209 | |g| 2.13 | 0.8M token | 6s | 136.4k tok/s
  b   200/6103 | loss 6.2632 | |g| 0.47 | 1.6M token | 12s | 139.9k tok/s
  b   300/6103 | loss 5.9351 | |g| 0.58 | 2.5M token | 18s | 139.9k tok/s
  b   400/6103 | loss 5.6230 | |g| 0.64 | 3.3M token | 24s | 138.6k tok/s
  b   500/6103 | loss 5.4489 | |g| 0.81 | 4.1M token | 30s | 137.9k tok/s
  --> val loss 5.4268 | val PPL 227.43
  b   600/6103 | loss 5.2576 | |g| 0.68 | 4.9M token | 38s | 127.9k tok/s
  b   700/6103 | loss 5.2226 | |g| 0.68 | 5.7M token | 45s | 128.9k tok/s
  b   800/6103 | los

In [30]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 1 --run_name E1_en_AAAA_s1 --out_dir results_t13


[E1_en_AAAA_s1] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_AAAA_s1] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_AAAA_s1] tham số: 7,383,552 (mixer 1,049,600 · mlp 2,102,272 · emb 4,096,000)
[E1_en_AAAA_s1] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8009 | |g| 2.25 | 0.8M token | 6s | 134.6k tok/s
  b   200/6103 | loss 6.2563 | |g| 0.94 | 1.6M token | 12s | 139.7k tok/s
  b   300/6103 | loss 5.4963 | |g| 0.93 | 2.5M token | 17s | 140.6k tok/s
  b   400/6103 | loss 5.5054 | |g| 0.75 | 3.3M token | 23s | 139.8k tok/s
  b   500/6103 | loss 5.3965 | |g| 0.65 | 4.1M token | 29s | 138.9k tok/s
  --> val loss 5.4273 | val PPL 227.53
  b   600/6103 | loss 5.5926 | |g| 0.71 | 4.9M token | 38s | 128.8k tok/s
  b   700/6103 | loss 5.3226 | |g| 0.73 | 5.7M token | 44s | 130.0k tok/s
  b   800/6103 | los

In [31]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 2 --run_name E1_en_AAAA_s2 --out_dir results_t13


[E1_en_AAAA_s2] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E1_en_AAAA_s2] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E1_en_AAAA_s2] tham số: 7,383,552 (mixer 1,049,600 · mlp 2,102,272 · emb 4,096,000)
[E1_en_AAAA_s2] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8493 | |g| 2.22 | 0.8M token | 6s | 135.9k tok/s
  b   200/6103 | loss 6.2603 | |g| 0.42 | 1.6M token | 12s | 139.7k tok/s
  b   300/6103 | loss 5.6062 | |g| 0.82 | 2.5M token | 18s | 140.0k tok/s
  b   400/6103 | loss 5.6711 | |g| 0.93 | 3.3M token | 23s | 139.5k tok/s
  b   500/6103 | loss 5.2686 | |g| 0.79 | 4.1M token | 30s | 138.7k tok/s
  --> val loss 5.4220 | val PPL 226.34
  b   600/6103 | loss 5.3575 | |g| 0.61 | 4.9M token | 38s | 128.5k tok/s
  b   700/6103 | loss 5.4729 | |g| 0.71 | 5.7M token | 44s | 129.6k tok/s
  b   800/6103 | los

## 5 · Kiểm và đóng gói

Mỗi file JSON phải ghi `corpus.n_tokens_train == 38.250.964`.


In [32]:
import json, glob

files = sorted(glob.glob("results_t13/E1_en_*.json"))
assert len(files) == 6, f"kỳ vọng 6 file JSON, thấy {len(files)}: {files}"
print(f"{'run':24s} {'train tokens':>14s} {'tokens seen':>13s} {'test PPL':>9s}")
for f in files:
    d = json.load(open(f))
    n = d["corpus"]["n_tokens_train"]
    assert n == 38250964, f"{f}: n_tokens_train={n} != 38250964 — cờ cắt không ăn!"
    print(f"{d['run_name']:24s} {n:>14,d} {d['tokens_seen']:>13,d} {d['test_ppl']:>9.3f}")
print("\nOK: cả 6 lần chạy đều cắt corpus đúng 38.250.964 token.")


run                        train tokens   tokens seen  test PPL
E1_en_AAAA_s0                38,250,964    49,995,776    65.595
E1_en_AAAA_s1                38,250,964    49,995,776    65.242
E1_en_AAAA_s2                38,250,964    49,995,776    65.447
E1_en_HHHH_s0                38,250,964    49,995,776    56.184
E1_en_HHHH_s1                38,250,964    49,995,776    55.498
E1_en_HHHH_s2                38,250,964    49,995,776    55.770

OK: cả 6 lần chạy đều cắt corpus đúng 38.250.964 token.


In [33]:
!zip -qr E1_en_matched_results.zip results_t13
try:
    from google.colab import files
    files.download("E1_en_matched_results.zip")
except Exception as e:
    print("Không tự tải được (", e, ")")
    print("-> Tải tay: panel Files bên trái -> E1_en_matched_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>